SLaM: Multi Wavelength Simultaneous
===================================

This example shows how to use the SLaM pipeline to fit a lens dataset at multiple wavelengths simultaneously.

Simultaneous multi-dataset fits are currently built into the SLaM pipeline without user input or customization.
Therefore, as long as lists of `Analysis` objects are created, summed and passed to the SLaM pipelines, the analysis
will fit every dataset simultaneously and it will adapt the model as follows:

- The first dataset is the astrometric reference and its sub-pixel offset is fixed at (0.0, 0.0). The sub-pixel
  offset of every other dataset is a free parameter in every stage of the pipeline, assuming broad uniform priors
  for every step. This is because the precision of a lens model can often be less than the requirements on
  astrometry.

- The regularization parameters are free for every dataset in the `source_pix[1]` and `source_pix[2]` stages. This is because
  the source morphology can be different between datasets, and the regularization scheme adapts to this.

- From the `light_lp` stage onwards, the regularization scheme for each dataset is different fixed to that inferred
  for the `source_pix[2]` stage.

Simultaneous fitting SLaM pipelines are not designed for customization, for example changing the model from the
set up above. This is because we are still figuring out the best way to perform multi-wavelength modeling, but have
so far figured the above settings are important.

If you need customization of the model or pipeline, you should pick apart the SLaM pipeline and customize
them as you see fit.

__Contents__

- **Preqrequisites:** Before reading this script, you should have familiarity with the following key concepts.
- **This Script:** Using a SOURCE LP PIPELINE, SOURCE PIX PIPELINE, LIGHT LP PIPELINE and TOTAL MASS PIPELINE this.
- **Dataset:** Load and plot the strong lens dataset.
- **Settings AutoFit:** The settings of autofit, which controls the output paths, parallelization, database use, etc.
- **Redshifts:** The redshifts of the lens and source galaxies.
- **SLaM Pipeline Functions:** Overview of slam pipeline functions for this example.
- **SLaM Pipeline:** Overview of slam pipeline for this example.

__Preqrequisites__

Before reading this script, you should have familiarity with the following key concepts:

- **Multi**: The `autolens_workspace/*/multi_dataset/features` package describes many different ways that multiple datasets
  can be modeled in a single analysis.

__This Script__

Using a SOURCE LP PIPELINE, SOURCE PIX PIPELINE, LIGHT LP PIPELINE and TOTAL MASS PIPELINE this SLaM modeling
script  fits `Imaging` dataset  of a strong lens system where in the final model:

 - The lens galaxy's light is a bulge with Multiple Gaussian Expansion (MGE) light profile.
 - The lens galaxy's total mass distribution is a `PowerLaw`; the external shear is an
   `ExternalShear` held in a `MassField`.
 - The source galaxy's light is a `Pixelization`.

This modeling script uses the SLaM pipelines:

 `source_lp`
 `source_pix`
 `light_lp`
 `mass_total`

__Start Here Notebook__

If any code in this script is unclear, refer to the `guides/modeling/slam_start_here.ipynb` notebook.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset__ 

Load, plot and mask the `Imaging` data.

In [ ]:
dataset_waveband_list = ["g", "r"]
pixel_scale_list = [0.12, 0.08]

dataset_name = "lens_sersic"
dataset_main_path = Path("dataset", "multi_dataset", "imaging", dataset_name)

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_main_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/multi_dataset/simulator.py"],
        check=True,
    )


dataset_list = []

for dataset_waveband, pixel_scale in zip(dataset_waveband_list, pixel_scale_list):
    dataset = al.Imaging.from_fits(
        data_path=Path(dataset_main_path, f"{dataset_waveband}_data.fits"),
        noise_map_path=Path(dataset_main_path, f"{dataset_waveband}_noise_map.fits"),
        psf_path=Path(dataset_main_path, f"{dataset_waveband}_psf.fits"),
        pixel_scales=pixel_scale,
    )

    mask_radius = 3.0

    mask = al.Mask2D.circular(
        shape_native=dataset.shape_native,
        pixel_scales=dataset.pixel_scales,
        radius=mask_radius,
    )

    dataset = dataset.apply_mask(mask=mask)

    over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
        grid=dataset.grid,
        sub_size_list=[4, 2, 2],
        radial_list=[0.3, 0.6],
        centre_list=[(0.0, 0.0)],
    )

    dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

    dataset_list.append(dataset)

__Settings AutoFit__

The settings of autofit, which controls the output paths, parallelization, database use, etc.

In [ ]:
settings_search = af.SettingsSearch(
    path_prefix=Path("slam", "multi_dataset", "simultaneous"),
    unique_tag=dataset_name,
    info=None,
    session=None,
)

__Redshifts__

The redshifts of the lens and source galaxies.

In [ ]:
redshift_lens = 0.5
redshift_source = 1.0

__SLaM Pipeline Functions__

In [ ]:


def source_lp(
    settings_search,
    analysis_list,
    lens_bulge,
    source_bulge,
    redshift_lens,
    redshift_source,
    dataset_model_list,
    mass_centre=(0.0, 0.0),
    n_batch=50,
):
    """
    SOURCE LP PIPELINE: fits an initial lens model using a parametric source, shared across all datasets
    via a factor graph so that the model parameters are the same for every waveband.
    """
    mass = af.Model(al.mp.Isothermal)
    mass.centre = mass_centre

    # The lens and source models are composed once, outside the loop below, so that their priors are shared
    # across every dataset. Only the `DatasetModel` varies from dataset to dataset.

    lens = af.Model(
        al.Galaxy,
        redshift=redshift_lens,
        bulge=lens_bulge,
        disk=None,
        mass=mass,
    )

    field = af.Model(
        al.MassField, redshift=redshift_lens, shear=af.Model(al.mp.ExternalShear)
    )

    source = af.Model(
        al.Galaxy,
        redshift=redshift_source,
        bulge=source_bulge,
    )

    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        model = af.Collection(
            galaxies=af.Collection(lens=lens, source=source),
            fields=field,
            dataset_model=dataset_model_list[i],
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="source_lp[1]",
        **settings_search.search_dict,
        n_live=200,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


def source_pix_1(
    settings_search,
    analysis_list,
    source_lp_result,
    mesh_shape,
    dataset_model_list,
    n_batch=20,
):
    """
    SOURCE PIX PIPELINE 1: initializes a pixelized source model, with per-dataset models that share
    the mass centre across wavebands to ensure consistent lens mass geometry.
    """
    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        mass = al.util.chaining.mass_from(
            mass=source_lp_result[i].model.galaxies.lens.mass,
            mass_result=source_lp_result[i].model.galaxies.lens.mass,
            unfix_mass_centre=True,
        )

        if i > 0:
            mass.centre = model.galaxies.lens.mass.centre

        model = af.Collection(
            galaxies=af.Collection(
                lens=af.Model(
                    al.Galaxy,
                    redshift=source_lp_result[i].instance.galaxies.lens.redshift,
                    bulge=source_lp_result[i].instance.galaxies.lens.bulge,
                    disk=source_lp_result[i].instance.galaxies.lens.disk,
                    mass=mass,
                ),
                source=af.Model(
                    al.Galaxy,
                    redshift=source_lp_result[i].instance.galaxies.source.redshift,
                    pixelization=af.Model(
                        al.Pixelization,
                        mesh=af.Model(
                            al.mesh.RectangularBilinearAdaptDensity, shape=mesh_shape
                        ),
                        regularization=al.reg.Adapt,
                    ),
                ),
            ),
            dataset_model=dataset_model_list[i],
            fields=source_lp_result[i].model.fields,
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="source_pix[1]",
        **settings_search.search_dict,
        n_live=150,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


def source_pix_2(
    settings_search,
    analysis_list,
    source_lp_result,
    source_pix_result_1,
    mesh_shape,
    dataset_model_list,
    n_batch=20,
):
    """
    SOURCE PIX PIPELINE 2: fits a pixelized source using adapt images from SOURCE PIX PIPELINE 1,
    with fixed lens mass and improved mesh and regularization.
    """
    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        model = af.Collection(
            galaxies=af.Collection(
                lens=af.Model(
                    al.Galaxy,
                    redshift=source_lp_result[i].instance.galaxies.lens.redshift,
                    bulge=source_lp_result[i].instance.galaxies.lens.bulge,
                    disk=source_lp_result[i].instance.galaxies.lens.disk,
                    mass=source_pix_result_1[i].instance.galaxies.lens.mass,
                ),
                source=af.Model(
                    al.Galaxy,
                    redshift=source_lp_result[i].instance.galaxies.source.redshift,
                    pixelization=af.Model(
                        al.Pixelization,
                        mesh=af.Model(
                            al.mesh.RectangularBilinearAdaptImage, shape=mesh_shape
                        ),
                        regularization=al.reg.Adapt,
                    ),
                ),
            ),
            dataset_model=dataset_model_list[i],
            fields=source_pix_result_1[i].instance.fields,
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="source_pix[2]",
        **settings_search.search_dict,
        n_live=75,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


def light_lp(
    settings_search,
    analysis_list,
    source_result_for_lens,
    source_result_for_source,
    lens_bulge,
    dataset_model_list,
    n_batch=20,
):
    """
    LIGHT LP PIPELINE: fits the lens galaxy light with mass and source fixed from SOURCE PIX PIPELINE,
    using per-dataset models sharing the same lens bulge model across wavebands.
    """
    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        source = al.util.chaining.source_custom_model_from(
            result=source_result_for_source[i], source_is_model=False
        )

        model = af.Collection(
            galaxies=af.Collection(
                lens=af.Model(
                    al.Galaxy,
                    redshift=source_result_for_lens[i].instance.galaxies.lens.redshift,
                    bulge=lens_bulge,
                    disk=None,
                    mass=source_result_for_lens[i].instance.galaxies.lens.mass,
                ),
                source=source,
            ),
            dataset_model=dataset_model_list[i],
            fields=source_result_for_lens[i].instance.fields,
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="light[1]",
        **settings_search.search_dict,
        n_live=250,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


def mass_total(
    settings_search,
    analysis_list,
    source_result_for_lens,
    source_result_for_source,
    light_result,
    dataset_model_list,
    n_batch=20,
):
    """
    MASS TOTAL PIPELINE: fits a PowerLaw total mass model using priors from SOURCE PIX PIPELINE,
    with lens light fixed from LIGHT LP PIPELINE.
    """
    # Total mass model for the lens galaxy.
    mass = af.Model(al.mp.PowerLaw)

    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        mass_i = al.util.chaining.mass_from(
            mass=mass,
            mass_result=source_result_for_lens[i].model.galaxies.lens.mass,
            unfix_mass_centre=True,
        )

        source = al.util.chaining.source_from(
            result=source_result_for_source[i],
        )

        model = af.Collection(
            galaxies=af.Collection(
                lens=af.Model(
                    al.Galaxy,
                    redshift=source_result_for_lens[i].instance.galaxies.lens.redshift,
                    bulge=light_result[i].instance.galaxies.lens.bulge,
                    disk=light_result[i].instance.galaxies.lens.disk,
                    mass=mass_i,
                ),
                source=source,
            ),
            dataset_model=dataset_model_list[i],
            fields=source_result_for_lens[i].model.fields,
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="mass_total[1]",
        **settings_search.search_dict,
        n_live=150,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


def subhalo_no_subhalo(
    settings_search,
    analysis_list,
    mass_result,
    dataset_model_list,
    n_batch=20,
):
    """
    SUBHALO PIPELINE 1: fits the lens model without a subhalo to provide Bayesian evidence for
    model comparison with the subhalo detection searches.
    """
    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        source = al.util.chaining.source_from(result=mass_result[i])
        lens = mass_result[i].model.galaxies.lens

        model = af.Collection(
            galaxies=af.Collection(lens=lens, source=source),
            fields=mass_result[i].model.fields,
            dataset_model=dataset_model_list[i],
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="subhalo[1]",
        **settings_search.search_dict,
        n_live=200,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


def subhalo_grid_search(
    settings_search,
    analysis_list,
    mass_result,
    subhalo_result_1,
    subhalo_mass,
    grid_dimension_arcsec=3.0,
    number_of_steps=2,
    n_batch=20,
):
    """
    SUBHALO PIPELINE 2: performs a grid search over subhalo positions to detect a dark matter subhalo,
    using per-dataset models with shared subhalo parameters.
    """
    subhalo = af.Model(al.Galaxy, mass=subhalo_mass)

    subhalo.mass.mass_at_200 = af.LogUniformPrior(lower_limit=1.0e6, upper_limit=1.0e11)
    subhalo.mass.centre_0 = af.UniformPrior(
        lower_limit=-grid_dimension_arcsec, upper_limit=grid_dimension_arcsec
    )
    subhalo.mass.centre_1 = af.UniformPrior(
        lower_limit=-grid_dimension_arcsec, upper_limit=grid_dimension_arcsec
    )

    subhalo.redshift = subhalo_result_1[0].instance.galaxies.lens.redshift
    subhalo.mass.redshift_object = subhalo_result_1[0].instance.galaxies.lens.redshift
    subhalo.mass.redshift_source = subhalo_result_1[0].instance.galaxies.source.redshift

    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        lens = mass_result[i].model.galaxies.lens
        source = al.util.chaining.source_from(result=mass_result[i])

        model = af.Collection(
            galaxies=af.Collection(lens=lens, subhalo=subhalo, source=source),
            fields=mass_result[i].model.fields,
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    search = af.Nautilus(
        name="subhalo[2]_[search_lens_plane]",
        **settings_search.search_dict,
        n_live=200,
        n_batch=n_batch,
    )

    subhalo_grid_search = af.SearchGridSearch(
        search=search,
        number_of_steps=number_of_steps,
    )

    return subhalo_grid_search.fit(
        model=model,
        analysis=analysis,
        grid_priors=[
            model.galaxies.subhalo.mass.centre_1,
            model.galaxies.subhalo.mass.centre_0,
        ],
        info=settings_search.info,
    )


def subhalo_refine(
    settings_search,
    analysis_list,
    subhalo_result_1,
    subhalo_grid_search_result_2,
    subhalo_mass,
    dataset_model_list,
    n_batch=20,
):
    """
    SUBHALO PIPELINE 3: refines the subhalo model parameters using priors initialized from the
    highest-evidence cell of the grid search.
    """
    subhalo = af.Model(
        al.Galaxy,
        redshift=subhalo_result_1[0].instance.galaxies.lens.redshift,
        mass=subhalo_mass,
    )

    subhalo.redshift = subhalo_result_1[0].instance.galaxies.lens.redshift
    subhalo.mass.redshift_object = subhalo_result_1[0].instance.galaxies.lens.redshift
    subhalo.mass.mass_at_200 = af.LogUniformPrior(lower_limit=1.0e6, upper_limit=1.0e11)
    subhalo.mass.centre = subhalo_grid_search_result_2.model_centred_absolute(
        a=1.0
    ).galaxies.subhalo.mass.centre
    subhalo.redshift = subhalo_grid_search_result_2.model.galaxies.subhalo.redshift
    subhalo.mass.redshift_object = subhalo.redshift

    analysis_factor_list = []

    for i, analysis in enumerate(analysis_list):
        model = af.Collection(
            galaxies=af.Collection(
                lens=subhalo_grid_search_result_2.model.galaxies.lens,
                subhalo=subhalo,
                source=subhalo_grid_search_result_2.model.galaxies.source,
            ),
            fields=subhalo_grid_search_result_2.model.fields,
            dataset_model=dataset_model_list[i],
        )

        analysis_factor = af.AnalysisFactor(prior_model=model, analysis=analysis)
        analysis_factor_list.append(analysis_factor)

    factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

    search = af.Nautilus(
        name="subhalo[3]_[single_plane_refine]",
        **settings_search.search_dict,
        n_live=600,
        n_batch=n_batch,
    )

    return search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)


__SLaM Pipeline__

In [ ]:

mesh_pixels_yx = 28
mesh_shape = (mesh_pixels_yx, mesh_pixels_yx)

__Dataset Offsets__

Sub-pixel offsets between the wavebands are modeled in every stage of the pipeline, via one `DatasetModel` per
dataset.

The first dataset is the astrometric reference, so its `DatasetModel` keeps the default offset of (0.0, 0.0),
which is a fixed value and not a free parameter. Every dataset after it has its offset freed with a broad uniform
prior, allowing its grid to shift by up to 1.0" in y and x relative to the first dataset.

Every stage below is passed this list and gives dataset `i` the model `dataset_model_list[i]`, so the offsets are
free parameters from the first search through to the final subhalo refinement.

In [ ]:
dataset_model_list = []

for i in range(len(dataset_list)):
    dataset_model = af.Model(al.DatasetModel)

    if i > 0:
        dataset_model.grid_offset.grid_offset_0 = af.UniformPrior(
            lower_limit=-1.0, upper_limit=1.0
        )
        dataset_model.grid_offset.grid_offset_1 = af.UniformPrior(
            lower_limit=-1.0, upper_limit=1.0
        )

    dataset_model_list.append(dataset_model)

# Lens Light

lens_bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=2,
    centre_prior_is_uniform=True,
    sigma_min=dataset.pixel_scales[0] / 10.0,
)

# Source Light

source_bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius, total_gaussians=20, centre_prior_is_uniform=False
)

analysis_list = [
    al.AnalysisImaging(dataset=dataset, use_jax=True) for dataset in dataset_list
]

source_lp_result = source_lp(
    settings_search=settings_search,
    analysis_list=analysis_list,
    lens_bulge=lens_bulge,
    source_bulge=source_bulge,
    redshift_lens=redshift_lens,
    redshift_source=redshift_source,
    dataset_model_list=dataset_model_list,
)

positions_likelihood = source_lp_result.positions_likelihood_from(
    factor=3.0, minimum_threshold=0.2
)

__Adapt Image S/N Cap__

The source adapt image is capped at a signal-to-noise of 3.0 before it is used by the adaptive
image-mesh and the adaptive regularization. Without the cap the brightest peak dominates the
weights (they scale as a power of the adapt image), so fainter multiply-imaged features get too
few source pixels and too little regularization weight. Capping makes every feature above S/N 3.0
count equally. The cap is applied to an explicit copy so the raw S/N image is untouched.

In [ ]:
adapt_images_list = []

for result in source_lp_result:
    galaxy_image_name_dict = al.galaxy_name_image_dict_via_result_from(result=result)

    # Cap the source adapt image at S/N 3.0 (see __Adapt Image S/N Cap__ above).
    adapt_image_snr_cap = 3.0

    source_adapt_image = galaxy_image_name_dict["('galaxies', 'source')"].copy()
    source_adapt_image[source_adapt_image > adapt_image_snr_cap] = adapt_image_snr_cap
    galaxy_image_name_dict["('galaxies', 'source')"] = source_adapt_image

    adapt_images = al.AdaptImages(galaxy_name_image_dict=galaxy_image_name_dict)
    adapt_images_list.append(adapt_images)

analysis_list = [
    al.AnalysisImaging(
        dataset=result.max_log_likelihood_fit.dataset,
        adapt_images=adapt_images,
        positions_likelihood_list=[positions_likelihood],
        use_jax=True,
    )
    for result, adapt_images in zip(source_lp_result, adapt_images_list)
]

source_pix_result_1 = source_pix_1(
    settings_search=settings_search,
    analysis_list=analysis_list,
    source_lp_result=source_lp_result,
    mesh_shape=mesh_shape,
    dataset_model_list=dataset_model_list,
)

__Pixelization Over-Sampling__

Every pixelized stage of this pipeline keeps the dataset's default uniform pixelization sub-size.

The single-dataset SLaM pipelines do something more refined. From SOURCE PIX PIPELINE 2 onwards they
threshold the source's signal-to-noise map at S/N 3.0 and pass the result as an adaptive
`over_sample_size_pixelization`: the bright lensed source gets a sub-size of 4 and the rest of the mask a
sub-size of 2, concentrating the over-sampling where the pixelization gains accuracy from it and leaving
the rest of the mask cheap. The sibling `independent.py` and
`imaging/advanced/subhalo/detect/start_here.py` both fit this way.

A simultaneous fit does not, because the saving does not survive the JAX compiler. A uniform sub-size bins
the over-sampled grid with one fixed-shape reshape and average; a non-uniform one has to bin ragged
sub-pixel blocks with a segmented sum, which costs far more to trace and compile even though it evaluates
fewer sub-pixels. No script pays that compilation more often than this one: once per SLaM stage, over every
band at once, and again for every cell of the SUBHALO grid search. Measured under the release-fidelity
validation environment, the adaptive map roughly tripled the SOURCE PIX PIPELINE 2 compile (12 s to 38 s)
and took the script from 367 s to past its 1800 s per-script cap, while the same change left the
single-dataset scripts faster.

If you adapt this script for your own multi-wavelength fit, keep the uniform sub-size and raise it before
you make it adaptive.

In [ ]:
adapt_images_list = []

for result in source_pix_result_1:
    galaxy_image_name_dict = al.galaxy_name_image_dict_via_result_from(result=result)

    # Cap the source adapt image at S/N 3.0 (see __Adapt Image S/N Cap__ above).
    adapt_image_snr_cap = 3.0

    source_adapt_image = galaxy_image_name_dict["('galaxies', 'source')"].copy()
    source_adapt_image[source_adapt_image > adapt_image_snr_cap] = adapt_image_snr_cap
    galaxy_image_name_dict["('galaxies', 'source')"] = source_adapt_image

    adapt_images = al.AdaptImages(galaxy_name_image_dict=galaxy_image_name_dict)
    adapt_images_list.append(adapt_images)

analysis_list = [
    al.AnalysisImaging(
        dataset=result.max_log_likelihood_fit.dataset,
        adapt_images=adapt_images,
        use_jax=True,
    )
    for result, adapt_images in zip(source_pix_result_1, adapt_images_list)
]

source_pix_result_2 = source_pix_2(
    settings_search=settings_search,
    analysis_list=analysis_list,
    source_lp_result=source_lp_result,
    source_pix_result_1=source_pix_result_1,
    mesh_shape=mesh_shape,
    dataset_model_list=dataset_model_list,
)

lens_bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=2,
    centre_prior_is_uniform=True,
    sigma_min=dataset.pixel_scales[0] / 10.0,
)

analysis_list = [
    al.AnalysisImaging(
        dataset=result.max_log_likelihood_fit.dataset,
        adapt_images=adapt_images,
        raise_inversion_positions_likelihood_exception=False,
    )
    for result, adapt_images in zip(source_pix_result_1, adapt_images_list)
]

light_result = light_lp(
    settings_search=settings_search,
    analysis_list=analysis_list,
    source_result_for_lens=source_pix_result_1,
    source_result_for_source=source_pix_result_2,
    lens_bulge=lens_bulge,
    dataset_model_list=dataset_model_list,
)

positions_likelihood = source_pix_result_1[0].positions_likelihood_from(
    factor=3.0, minimum_threshold=0.2
)

analysis_list = [
    al.AnalysisImaging(
        dataset=result.max_log_likelihood_fit.dataset,
        adapt_images=adapt_images,
        positions_likelihood_list=[positions_likelihood],
    )
    for result, adapt_images in zip(source_pix_result_1, adapt_images_list)
]

mass_result = mass_total(
    settings_search=settings_search,
    analysis_list=analysis_list,
    source_result_for_lens=source_pix_result_1,
    source_result_for_source=source_pix_result_2,
    light_result=light_result,
    dataset_model_list=dataset_model_list,
)

subhalo_result_1 = subhalo_no_subhalo(
    settings_search=settings_search,
    analysis_list=analysis_list,
    mass_result=mass_result,
    dataset_model_list=dataset_model_list,
)

subhalo_grid_search_result_2 = subhalo_grid_search(
    settings_search=settings_search,
    analysis_list=analysis_list,
    mass_result=mass_result,
    subhalo_result_1=subhalo_result_1,
    subhalo_mass=af.Model(al.mp.NFWMCRLudlowSph),
    grid_dimension_arcsec=3.0,
    number_of_steps=2,
)

subhalo_result_3 = subhalo_refine(
    settings_search=settings_search,
    analysis_list=analysis_list,
    subhalo_result_1=subhalo_result_1,
    subhalo_grid_search_result_2=subhalo_grid_search_result_2,
    subhalo_mass=af.Model(al.mp.NFWMCRLudlowSph),
    dataset_model_list=dataset_model_list,
)
